1. Install and import required libraries

In [1]:
%pip install pandas pyreadr joblib beautifulsoup4 scikit-learn

import os
import re
import joblib
import pyreadr

from bs4 import BeautifulSoup
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


2. Read RData into pandas dataframes

In [2]:
raw_data = pyreadr.read_r("../data/claims-raw.RData")
claims_raw = raw_data["claims_raw"]

test_data = pyreadr.read_r("../data/claims-test.RData")
claims_test = test_data["claims_test"]

3. Data preprocessing (turning the HTML into raw words)

In [ ]:
def clean_html(html):
    soup = BeautifulSoup(str(html), "html.parser")

    elements = soup.find_all([
        "p", "h1", "h2", "h3", "h4", "h5", "h6"
    ])

    text = " ".join(element.get_text(" ") for element in elements)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


claims_raw["clean_text"] = claims_raw["text_tmp"].apply(clean_html)
claims_test["clean_text"] = claims_test["text_tmp"].apply(clean_html)

4. Train the model (first extract TF-IDF, then traing logistic regression using 5 fold stratified cross validation)

In [4]:
X = claims_raw["clean_text"]
y = claims_raw["mclass"]

X_train, X_validation, y_train, y_validation = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=123
)

# Model pipeline
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000
    )),
    ("logistic_regression", LogisticRegression(
        max_iter=2000,
    ))
])


# Five-fold cross-validation AUROC
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc_ovr",
    n_jobs=-1
)

print(f"Mean CV AUROC: {scores.mean():.3f}")

Mean CV AUROC: 0.941


5. Report the model performance based on validation set

In [5]:
model.fit(X_train, y_train)

valid_pred = model.predict(X_validation)
valid_prob = model.predict_proba(X_validation)

classes = model.classes_
cm = confusion_matrix(y_validation, valid_pred, labels=classes)

sensitivities = []
specificities = []

for i in range(len(classes)):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp

    sensitivities.append(tp / (tp + fn))
    specificities.append(tn / (tn + fp))

print("Accuracy:", accuracy_score(y_validation, valid_pred))
print("Macro sensitivity:", np.mean(sensitivities))
print("Macro specificity:", np.mean(specificities))

Accuracy: 0.7990762124711316
Macro sensitivity: 0.6695024438972916
Macro specificity: 0.9325097829501734


6. Save the model, and model's predictions on test set

In [ ]:
# Refit on all labeled pages (train + validation) for the final model
model.fit(X, y)

test_predictions = model.predict(claims_test["clean_text"])

pred_df = claims_test[[".id"]].copy()
pred_df["mclass.pred"] = test_predictions

pyreadr.write_rdata(
    "../results/test_set_multiclass_predictions.RData",
    pred_df,
    df_name="pred_df"
)

joblib.dump(
    model,
    "../results/multiclass_model.joblib"
)